In [ ]:
import json
import os
import time
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError, field_validator

TASK 4.1 One validated call

In [48]:
load_dotenv()
api_key = os.getenv("API_KEY")
if not api_key:
    raise ValueError("API_KEY not found in .env file")


In [49]:
client = genai.Client(api_key=api_key)
model_name = "gemini-3.5-flash"

In [50]:
allowed_causes = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

schema = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

system_text = (
    "You read clinic call-centre notes and return structured JSON with "
    "root_cause, confidence, summary, and recommended_action."
)

calls_made = 0


In [51]:
def make_prompt(note_text):
    causes_text = ", ".join(allowed_causes)
    return (
        "Context: A clinic call-centre wrote short notes on why patients missed appointments "
        "(some notes may be in Hinglish or unclear).\n"
        "Role: You are a clinic support analyst.\n"
        f"Action: Pick one root_cause from [{causes_text}]. If the reason is unclear or "
        "ambiguous, pick 'other' instead of guessing. Give a confidence score between 0 and 1, "
        "a short summary of at most 25 words, and a recommended_action.\n"
        "Tone: Clear, simple, and practical.\n"
        "Format: Return JSON matching the schema.\n\n"
        f"Note: {note_text}"
    )

In [52]:
def analyse_note(note_text, custom_prompt=None):
    global calls_made
    prompt = custom_prompt if custom_prompt is not None else make_prompt(note_text)

    resp = client.models.generate_content(
        model=model_name,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_text,
            temperature=0,
            max_output_tokens=1200,
            response_mime_type="application/json",
            response_schema=schema,
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),
        ),
    )
    calls_made += 1
    return json.loads(resp.text)


In [53]:
notes_df = pd.read_csv("followup_notes.csv")
n001_note = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].iloc[0]
n001_result = analyse_note(n001_note)
print("4.1 N001 result:", n001_result)

4.1 N001 result: {'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'Patient forgot the appointment because no reminder message was received. Requested to reschedule for next week.', 'recommended_action': 'Reschedule appointment for next week and verify patient contact details to ensure automated SMS/email reminders are enabled.'}


TASK 4.2 Validate and process the 10 notes

In [54]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

    @field_validator("root_cause")
    @classmethod
    def check_allowed(cls, v):
        if v not in allowed_causes:
            raise ValueError("root_cause not in allowed list")
        return v

In [55]:
def is_empty_note(val):
    if pd.isna(val):
        return True
    return str(val).strip() == ""


In [56]:
saved_rows = []
first_10 = notes_df.head(10)

for _, row in first_10.iterrows():
    nid = row["note_id"]
    aid = row["appointment_id"]
    text = row["note_text"]

    if is_empty_note(text):
        saved_rows.append(
            {
                "note_id": nid,
                "appointment_id": aid,
                "root_cause": "other",
                "confidence": 0.0,
                "summary": "No note recorded",
                "recommended_action": "Re-call patient",
                "validation_status": "skipped_blank",
                "raw_response": "",
            }
        )
        continue

    if nid != "N001":
        time.sleep(13)

    raw_text = ""
    parsed = {}
    try:
        # reuse N001 result from Task 4.1 so we don't make an extra call
        parsed = n001_result if nid == "N001" else analyse_note(str(text))
        raw_text = json.dumps(parsed)
        FollowupInsight(**parsed)
        status = "valid"
    except Exception:
        status = "invalid"
        if not raw_text and isinstance(parsed, dict) and len(parsed) > 0:
            raw_text = json.dumps(parsed)

    saved_rows.append(
        {
            "note_id": nid,
            "appointment_id": aid,
            "root_cause": parsed.get("root_cause", "") if isinstance(parsed, dict) else "",
            "confidence": parsed.get("confidence", "") if isinstance(parsed, dict) else "",
            "summary": parsed.get("summary", "") if isinstance(parsed, dict) else "",
            "recommended_action": (
                parsed.get("recommended_action", "") if isinstance(parsed, dict) else ""
            ),
            "validation_status": status,
            "raw_response": raw_text,
        }
    )

analysed_df = pd.DataFrame(saved_rows)
analysed_df.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv:")
print(analysed_df[["note_id", "root_cause", "confidence", "validation_status"]])


Saved notes_analysed.csv:
  note_id             root_cause confidence validation_status
0    N001  forgot_or_no_reminder       0.95             valid
1    N002  distance_or_transport       0.98             valid
2    N003            cost_or_fee       0.98             valid
3    N004                  other        0.0     skipped_blank
4    N005   illness_or_emergency       0.95             valid
5    N006                                             invalid
6    N007                                             invalid
7    N008                                             invalid
8    N009                  other        0.9             valid
9    N010       work_or_schedule       0.95             valid


TASK 4.3 Break it on purpose

In [57]:
bad_conf = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "Patient missed visit.",
    "recommended_action": "Call patient.",
}
bad_missing = {
    "root_cause": "other",
    "confidence": 0.7,
    "summary": "Patient missed visit.",
}

for label, bad_item in [("confidence = 1.7", bad_conf), ("missing recommended_action", bad_missing)]:
    try:
        FollowupInsight(**bad_item)
    except ValidationError as err:
        print(f"\n {label} error:\n{err}")



 confidence = 1.7 error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

 missing recommended_action error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...'Patient missed visit.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [58]:
v1_rows = []
v2_lookup = dict(zip(analysed_df["note_id"], analysed_df["root_cause"]))

for target_id in ["N007", "N009"]:
    target_text = str(first_10.loc[first_10["note_id"] == target_id, "note_text"].iloc[0])

    time.sleep(13)
    out_v1 = analyse_note(target_text, custom_prompt=f"Classify this note:\n{target_text}")
    rc_v1 = out_v1.get("root_cause", "")

    # reuse V2 result already obtained in 4.2
    rc_v2 = v2_lookup.get(target_id, "")
    if rc_v2 not in allowed_causes:
        time.sleep(13)
        out_v2 = analyse_note(target_text, custom_prompt=make_prompt(target_text))
        rc_v2 = out_v2.get("root_cause", "")

    v1_rows.append(
        {
            "note": target_id,
            "v1_cause": rc_v1,
            "v1_ok": rc_v1 in allowed_causes,
            "v2_cause": rc_v2,
            "v2_ok": rc_v2 in allowed_causes,
        }
    )

print("\n V1 vs V2:")
print(pd.DataFrame(v1_rows))
print("I would deploy V2 because it lists the allowed root causes and tells the model to pick 'other' instead of guessing.")


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
print(f"\n4.3 Total API calls made: {calls_made}")


4.3 Total API calls made: 11


TASK 4.4 Recommendation and GitHub

In [ ]:
appt_df = pd.read_csv("appointments_clean.csv")
merged_df = analysed_df.merge(appt_df, on="appointment_id", how="left")
print("\n4.4(a) Merged rows:", len(merged_df))
print(pd.crosstab(merged_df["root_cause"], merged_df["reminder_sent"], margins=True))


4.4(a) Merged rows: 10
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   1    1    2
forgot_or_no_reminder   2    0    2
illness_or_emergency    0    1    1
other                   1    1    2
work_or_schedule        1    1    2
All                     5    5   10
